In [ ]:
from shiny import App, ui, render, run_app
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import asyncio

# --- Se Cargan datos ---
base = pd.read_csv("./base_de_datos_cantones.csv")

# --- Se Define interfaz con pestañas ---
app_ui = ui.page_fluid(
    ui.h2("Análisis Socioeconómico de Cantones de Costa Rica"),
    
    ui.navset_pill(
        # --- Pestaña 1: Gráfico principal ---
        ui.nav_panel(
            "Relación Desempleo-Pobreza",
            ui.layout_columns(
                ui.card(
                    ui.input_select(
                        "provincia",
                        "Seleccionar provincia:",
                        choices=["Todas"] + sorted(base["provincia"].unique().tolist()),
                        selected="Todas"
                    ),
                    ui.input_slider(
                        "rango_des",
                        "Filtrar por tasa de desempleo abierto (%)",
                        min=base["Tasa de desempleo abierto"].min(),
                        max=base["Tasa de desempleo abierto"].max(),
                        value=(
                            base["Tasa de desempleo abierto"].min(),
                            base["Tasa de desempleo abierto"].max()
                        ),
                        step=0.1
                    ),
                    ui.input_slider(
                        "rango_pobreza",
                        "Filtrar por porcentaje de hogares pobres (%)",
                        min=base["Porcentaje de hogares pobres"].min(),
                        max=base["Porcentaje de hogares pobres"].max(),
                        value=(
                            base["Porcentaje de hogares pobres"].min(),
                            base["Porcentaje de hogares pobres"].max()
                        ),
                        step=0.1
                    )
                ),
                ui.card(
                    ui.output_plot("grafico", width="100%", height="500px"),
                    ui.output_text("resumen")
                )
            )
        ),
        
        # --- Pestaña 2: Visualización adicional 1 - Sectores económicos ---
        ui.nav_panel(
            "Sectores Económicos",
            ui.layout_columns(
                ui.card(
                    ui.input_select(
                        "provincia_sectores",
                        "Seleccionar provincia:",
                        choices=["Todas"] + sorted(base["provincia"].unique().tolist()),
                        selected="Todas"
                    ),
                    ui.input_select(
                        "sector",
                        "Seleccionar sector económico:",
                        choices={
                            "Sector primario": "Sector primario",
                            "Sector secundario": "Sector secundario", 
                            "Sector terciario": "Sector terciario"
                        },
                        selected="Sector primario"
                    )
                ),
                ui.card(
                    ui.output_plot("grafico_sectores", width="100%", height="500px")
                )
            )
        ),
        
        # --- Pestaña 3: Visualización adicional 2 - Ranking cantones ---
        ui.nav_panel(
            "Ranking Cantones",
            ui.layout_columns(
                ui.card(
                    ui.input_select(
                        "variable_ranking",
                        "Seleccionar variable para ranking:",
                        choices={
                            "Tasa de desempleo abierto": "Tasa de desempleo abierto",
                            "Porcentaje de hogares pobres": "Porcentaje de hogares pobres",
                            "Tasa bruta de natalidad": "Tasa bruta de natalidad"
                        },
                        selected="Tasa de desempleo abierto"
                    ),
                    ui.input_slider(
                        "top_n",
                        "Número de cantones a mostrar:",
                        min=5,
                        max=20,
                        value=10,
                        step=1
                    )
                ),
                ui.card(
                    ui.output_plot("grafico_ranking", width="100%", height="500px")
                )
            )
        ),
        
        # --- Pestaña 4: Descripción de datos ---
        ui.nav_panel(
            "Descripción de Datos",
            ui.card(
                ui.h3("Información del Conjunto de Datos"),
                ui.p("Este conjunto de datos contiene información socioeconómica de los 81 cantones de Costa Rica."),
                ui.h4("Variables principales:"),
                ui.p("• Población y demografía"),
                ui.p("• Tasas de empleo y desempleo"), 
                ui.p("• Pobreza y distribución económica"),
                ui.p("• Sectores económicos (primario, secundario, terciario)"),
                ui.p("• Natalidad y mortalidad"),
                ui.h4("Resumen estadístico:"),
                ui.output_table("tabla_resumen")
            )
        )
    )
)

# --- Definir servidor ---
def server(input, output, session):

    # --- Gráfico original (dispersión) ---
    @output
    @render.plot
    def grafico():
        # Filtrar datos
        if input.provincia() == "Todas":
            df = base.copy()
        else:
            df = base[base["provincia"] == input.provincia()]

        df_filtrado = df[
            (df["Tasa de desempleo abierto"] >= input.rango_des()[0]) &
            (df["Tasa de desempleo abierto"] <= input.rango_des()[1]) &
            (df["Porcentaje de hogares pobres"] >= input.rango_pobreza()[0]) &
            (df["Porcentaje de hogares pobres"] <= input.rango_pobreza()[1])
        ]

        # Crear gráfico
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(
            df_filtrado["Tasa de desempleo abierto"],
            df_filtrado["Porcentaje de hogares pobres"],
            color="teal",
            alpha=0.7,
            edgecolors="black"
        )

        # Línea de tendencia
        x = df_filtrado["Tasa de desempleo abierto"]
        y = df_filtrado["Porcentaje de hogares pobres"]

        if len(df_filtrado) > 1:
            coef = np.polyfit(x, y, 1)
            tendencia = np.poly1d(coef)
            ax.plot(x, tendencia(x), color="black", linewidth=2, label="Tendencia lineal")
            ax.legend()

            # Coeficiente de correlación
            corr = x.corr(y)
            ax.text(
                0.05, 0.95, f"r = {corr:.2f}",
                transform=ax.transAxes,
                fontsize=10,
                color="black",
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.6)
            )

        # Etiquetas de cantones
        for _, row in df_filtrado.iterrows():
            ax.text(
                row["Tasa de desempleo abierto"],
                row["Porcentaje de hogares pobres"],
                row["Cantón"],
                fontsize=7,
                alpha=0.7
            )

        # Estilo
        ax.set_xlabel("Tasa de desempleo abierto (%)")
        ax.set_ylabel("Porcentaje de hogares pobres (%)")
        ax.set_title(f"Relación entre desempleo y pobreza por cantón\n({input.provincia()})")
        ax.grid(True, linestyle="--", alpha=0.4)
        plt.tight_layout()
        return fig

    @output
    @render.text
    def resumen():
        if input.provincia() == "Todas":
            df = base.copy()
        else:
            df = base[base["provincia"] == input.provincia()]

        df_filtrado = df[
            (df["Tasa de desempleo abierto"] >= input.rango_des()[0]) &
            (df["Tasa de desempleo abierto"] <= input.rango_des()[1]) &
            (df["Porcentaje de hogares pobres"] >= input.rango_pobreza()[0]) &
            (df["Porcentaje de hogares pobres"] <= input.rango_pobreza()[1])
        ]

        if len(df_filtrado) > 1:
            corr = df_filtrado["Tasa de desempleo abierto"].corr(df_filtrado["Porcentaje de hogares pobres"])
            interpretacion = (
                "Existe una fuerte relación positiva entre desempleo y pobreza."
                if corr > 0.5 else
                "Existe una relación débil o moderada entre desempleo y pobreza."
                if corr > 0.2 else
                "No se observa una relación clara entre desempleo y pobreza."
            )
            return f"Correlación (r): {corr:.2f}. {interpretacion} Cantones mostrados: {len(df_filtrado)}"
        else:
            return "Insuficientes datos para calcular la correlación."

    # --- Visualización adicional 1: Sectores económicos ---
    @output
    @render.plot
    def grafico_sectores():
        if input.provincia_sectores() == "Todas":
            df = base.copy()
        else:
            df = base[base["provincia"] == input.provincia_sectores()]

        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Ordenar por el sector seleccionado
        df_sorted = df.sort_values(input.sector(), ascending=False)
        
        bars = ax.barh(
            df_sorted["Cantón"], 
            df_sorted[input.sector()],
            color="skyblue",
            edgecolor="navy"
        )
        
        ax.set_xlabel(f"{input.sector()} (%)")
        ax.set_ylabel("Cantón")
        ax.set_title(f"Distribución del {input.sector()} por cantón\n({input.provincia_sectores()})")
        ax.grid(True, linestyle="--", alpha=0.3, axis="x")
        plt.tight_layout()
        return fig

    # --- Visualización adicional 2: Ranking cantones ---
    @output
    @render.plot
    def grafico_ranking():
        df_sorted = base.sort_values(input.variable_ranking(), ascending=False).head(input.top_n())
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        bars = ax.barh(
            df_sorted["Cantón"], 
            df_sorted[input.variable_ranking()],
            color="lightcoral",
            edgecolor="darkred"
        )
        
        # Añadir valores en las barras
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width, bar.get_y() + bar.get_height()/2, 
                   f'{width:.1f}', ha='left', va='center')
        
        ax.set_xlabel(input.variable_ranking())
        ax.set_ylabel("Cantón")
        ax.set_title(f"Top {input.top_n()} cantones por {input.variable_ranking()}")
        ax.grid(True, linestyle="--", alpha=0.3, axis="x")
        plt.tight_layout()
        return fig

    # --- Tabla de resumen para pestaña de descripción ---
    @output
    @render.table
    def tabla_resumen():
        # Estadísticas básicas de variables numéricas
        variables = ["Tasa de desempleo abierto", "Porcentaje de hogares pobres", 
                    "Sector primario", "Sector secundario", "Sector terciario"]
        
        resumen_data = []
        for var in variables:
            resumen_data.append({
                "Variable": var,
                "Mínimo": f"{base[var].min():.2f}",
                "Máximo": f"{base[var].max():.2f}",
                "Promedio": f"{base[var].mean():.2f}",
                "Cantones": len(base)
            })
        
        return pd.DataFrame(resumen_data)

# --- Crear la app ---
app = App(app_ui, server)

# --- Para ejecutar---
await asyncio.to_thread(run_app, app, port=8010)

INFO:     Started server process [13738]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8010 (Press CTRL+C to quit)


INFO:     127.0.0.1:51319 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51321 - "GET /lib/shiny-1.5.0/shiny.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51320 - "GET /lib/jquery-3.6.0/jquery-3.6.0.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51319 - "GET /lib/requirejs-2.3.6/require.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51324 - "GET /lib/bootstrap-5.3.1/bootstrap.bundle.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51323 - "GET /lib/bootstrap-5.3.1/bootstrap.min.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:51322 - "GET /lib/shiny-busy-indicators-1.5.0/busy-indicators.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:51322 - "GET /lib/bslib-components-0.9.0.9000/components.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51319 - "GET /lib/bslib-components-0.9.0.9000/web-components.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51321 - "GET /lib/bootstrap-datepicker-1.9.0/js/bootstrap-datepicker.min.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51320 - "GET /lib/strftime-0.9.2/strftime-min.js HTTP/1.1" 200 OK
INFO: 

INFO:     ('127.0.0.1', 51325) - "WebSocket /websocket/" [accepted]
INFO:     connection open


INFO:     127.0.0.1:51322 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     connection closed
